# Zero-Day Attack Classification — UNSW-NB15 (Anonymized Ablation)

Replication of the WUSTL-IIoT anonymized ablation on the UNSW-NB15 dataset.
Run **twice**:

1. **Original**: real feature names (`dur`, `spkts`, `sbytes`, …)
2. **Anonymized**: opaque labels `f0`, `f1`, … `f{n}`

**Hypothesis:** If feature-name priors drive zero-day generalisation, anonymizing names should collapse ZDR to DT/RF levels (~0–10%).

**Data source:** Two pre-split CSV files concatenated into a single population before applying the same zero-day split logic as WUSTL.

In [1]:
################################################################################
# Cell 0 - Configuration
################################################################################

# === Change WITHHELD_CLASS to run a different zero-day scenario ===
#
# Candidate classes for UNSW-NB15 (approximate combined sample counts):
#   'Reconnaissance' ~13,987 samples - RECOMMENDED (scanning paradigm, distinct from exploitation)
#   'DoS'            ~16,353 samples - volumetric stress test
#   'Fuzzers'        ~24,246 samples - high support
#   'Analysis'        ~2,677 samples - low support; exploratory only
#   'Backdoor'        ~2,329 samples - low support; exploratory only
#
# Selection rationale:
# Reconnaissance matches WUSTL Reconn conceptually (scanning/probing paradigm),
# enabling direct cross-dataset comparison of the anonymization effect.

WITHHELD_CLASS  = 'Reconnaissance'

TRAIN_PATH      = '/Users/S4160163/Documents/Projects/RAG Paper/data/unsw-nb15/UNSW_NB15_training-set.csv'
TEST_PATH       = '/Users/S4160163/Documents/Projects/RAG Paper/data/unsw-nb15/UNSW_NB15_testing-set.csv'
DATASET_NAME    = 'unsw-nb15'
LABEL_COL       = 'attack_cat'    # multiclass label column
BENIGN_CLASS    = 'Normal'        # benign class name (case-sensitive)

# Columns to exclude from features
# 'id' is a numeric record identifier — not a network feature
DROP_COLS       = ['label', 'attack_cat', 'id']

N_REPR          = 10       # representative samples per class fed to LLM
MAX_EMBED       = 100      # max rows to embed per class
k               = 5        # number of rules
n               = 5        # number of feedback iterations
SEEDS           = [42, 123, 456]
SEED_MAIN       = 42

print(f'Withheld (zero-day) class : {WITHHELD_CLASS}')
print(f'Benign class              : {BENIGN_CLASS}')
print(f'Label column              : {LABEL_COL}')
print(f'Rules: {k} | Iterations: {n} | Seeds: {SEEDS}')

Withheld (zero-day) class : Reconnaissance
Benign class              : Normal
Label column              : attack_cat
Rules: 5 | Iterations: 5 | Seeds: [42, 123, 456]


In [2]:
################################################################################
# Cell 0b — Anonymization map
#
# Built AFTER feature_cols is known (Cell 1 loads data and sets feature_cols).
# anon_map   : real_name  → f{i}
# reverse_map: f{i}       → real_name
################################################################################

def build_anon_maps(cols):
    a = {name: f'f{i}' for i, name in enumerate(cols)}
    r = {f'f{i}': name for i, name in enumerate(cols)}
    return a, r

print('build_anon_maps() ready — call after Cell 1 has set feature_cols.')

build_anon_maps() ready — call after Cell 1 has set feature_cols.


In [3]:
################################################################################
# Cell 1 - Load data
#
# Two source files (training + testing) are concatenated into a single population
# DataFrame before applying the same zero-day split logic as the WUSTL notebook.
# encoding='utf-8-sig' handles the BOM marker present in both CSV headers.
################################################################################

import pandas as pd
import numpy as np
import os
from tabulate import tabulate

df_train = pd.read_csv(TRAIN_PATH, encoding='utf-8-sig')
df_test  = pd.read_csv(TEST_PATH,  encoding='utf-8-sig')
df_raw   = pd.concat([df_train, df_test], ignore_index=True)

# Keep only numeric feature columns (drop label and identifier columns)
feature_cols = [
    c for c in df_raw.select_dtypes(include=[np.number]).columns
    if c not in DROP_COLS
]
df = df_raw[feature_cols + [LABEL_COL]].copy()

# Strip any accidental whitespace from class labels
df[LABEL_COL] = df[LABEL_COL].str.strip()

print(f'=== UNSW-NB15 Population ===')
print(f'Training file : {TRAIN_PATH}')
print(f'Testing file  : {TEST_PATH}')
print(f'Training rows : {len(df_train):,}')
print(f'Testing rows  : {len(df_test):,}')
print(f'Total rows    : {len(df):,}')
print(f'Features      : {len(feature_cols)}  ->  {feature_cols}')
print()

label_counts = df[LABEL_COL].value_counts()
print('Class distribution:')
print(label_counts.to_string())
print()

benign_count   = label_counts.get(BENIGN_CLASS, 0)
withheld_count = label_counts.get(WITHHELD_CLASS, 0)
print(f'Benign ("{BENIGN_CLASS}") rows   : {benign_count:,}')
print(f'Withheld ("{WITHHELD_CLASS}") rows: {withheld_count:,}')
print(f'\nTraining will be balanced to the smaller class size (1:1).')

assert WITHHELD_CLASS in df[LABEL_COL].unique(), f'{WITHHELD_CLASS} not found in {LABEL_COL}'

=== UNSW-NB15 Population ===
Training file : /Users/S4160163/Documents/Projects/RAG Paper/data/unsw-nb15/UNSW_NB15_training-set.csv
Testing file  : /Users/S4160163/Documents/Projects/RAG Paper/data/unsw-nb15/UNSW_NB15_testing-set.csv
Training rows : 175,341
Testing rows  : 82,332
Total rows    : 257,673
Features      : 39  ->  ['dur', 'spkts', 'dpkts', 'sbytes', 'dbytes', 'rate', 'sttl', 'dttl', 'sload', 'dload', 'sloss', 'dloss', 'sinpkt', 'dinpkt', 'sjit', 'djit', 'swin', 'stcpb', 'dtcpb', 'dwin', 'tcprtt', 'synack', 'ackdat', 'smean', 'dmean', 'trans_depth', 'response_body_len', 'ct_srv_src', 'ct_state_ttl', 'ct_dst_ltm', 'ct_src_dport_ltm', 'ct_dst_sport_ltm', 'ct_dst_src_ltm', 'is_ftp_login', 'ct_ftp_cmd', 'ct_flw_http_mthd', 'ct_src_ltm', 'ct_srv_dst', 'is_sm_ips_ports']

Class distribution:
attack_cat
Normal            93000
Generic           58871
Exploits          44525
Fuzzers           24246
DoS               16353
Reconnaissance    13987
Analysis           2677
Backdoor    

In [4]:
################################################################################
# Cell 2 — Prepare zero-day split
#
# Balancing strategy:
#   - Downsample the larger class so benign and known attacks are exactly matched
#   - 1:1 pool to avoid imbalance-driven bias
#   - Withheld class excluded from training pool entirely
#
# Zero-day test set:
#   - ALL rows of WITHHELD_CLASS (never in training, never in test_known)
################################################################################

benign_df    = df[df[LABEL_COL] == BENIGN_CLASS].copy()
known_atk_df = df[
    (df[LABEL_COL] != BENIGN_CLASS) & (df[LABEL_COL] != WITHHELD_CLASS)
].copy()
zeroday_df   = df[df[LABEL_COL] == WITHHELD_CLASS].copy()

assert len(zeroday_df) > 0, 'Zero-day set is empty'

print(f'Benign rows   : {len(benign_df):,}')
print(f'Known attack  : {len(known_atk_df):,}  ({known_atk_df[LABEL_COL].nunique()} classes: {sorted(known_atk_df[LABEL_COL].unique())})')
print(f'Zero-day rows : {len(zeroday_df):,}  ("{WITHHELD_CLASS}")')

# Balance: sample both classes to the same target count (1:1)
target_count = min(len(benign_df), len(known_atk_df))
benign_sampled     = benign_df.sample(n=target_count, random_state=SEED_MAIN)
known_atk_sampled  = known_atk_df.sample(n=target_count, random_state=SEED_MAIN)
print(f'\nBalanced pool: {target_count:,} benign + {target_count:,} known attack (1:1)')

# Binary labels
train_pool = pd.concat([benign_sampled, known_atk_sampled], ignore_index=True)
train_pool['binary_label'] = train_pool[LABEL_COL].apply(
    lambda x: 'normal' if x == BENIGN_CLASS else 'attack'
)

# Structural leakage check: withheld class must be absent from the training pool by label
assert WITHHELD_CLASS not in train_pool[LABEL_COL].values, 'Withheld class leaked into train pool'

# Stratified 80/20 split
from sklearn.model_selection import train_test_split
train_idx, test_idx = train_test_split(
    train_pool.index, test_size=0.2, random_state=SEED_MAIN,
    stratify=train_pool['binary_label']
)
train_df      = train_pool.loc[train_idx]
test_known_df = train_pool.loc[test_idx]

# Feature-only dataframes (global — referenced by evaluation_tool and evaluate_node)
normal_df_train = train_df[train_df['binary_label'] == 'normal'][feature_cols].reset_index(drop=True)
attack_df_train = train_df[train_df['binary_label'] == 'attack'][feature_cols].reset_index(drop=True)
normal_df_test  = test_known_df[test_known_df['binary_label'] == 'normal'][feature_cols].reset_index(drop=True)
attack_df_test  = test_known_df[test_known_df['binary_label'] == 'attack'][feature_cols].reset_index(drop=True)

# Zero-day test = ALL withheld class rows (guaranteed absent from training)
zeroday_df_test = zeroday_df[feature_cols].reset_index(drop=True)

data = [
    ['Normal (train)',                        len(normal_df_train)],
    ['Known attack (train)',                  len(attack_df_train)],
    ['Normal (test, known)',                  len(normal_df_test)],
    ['Known attack (test, known)',            len(attack_df_test)],
    [f'Zero-day test ({WITHHELD_CLASS})',     len(zeroday_df_test)],
]
print('\nDataset splits:')
print(tabulate(data, headers=['Split', 'Count'], tablefmt='grid'))
print(f'\nFeatures : {len(feature_cols)}')
print('Zero-day samples are NOT present in training.')

Benign rows   : 93,000
Known attack  : 150,686  (8 classes: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Shellcode', 'Worms'])
Zero-day rows : 13,987  ("Reconnaissance")

Balanced pool: 93,000 benign + 93,000 known attack (1:1)

Dataset splits:
+--------------------------------+---------+
| Split                          |   Count |
+================================+=========+
| Normal (train)                 |   74400 |
+--------------------------------+---------+
| Known attack (train)           |   74400 |
+--------------------------------+---------+
| Normal (test, known)           |   18600 |
+--------------------------------+---------+
| Known attack (test, known)     |   18600 |
+--------------------------------+---------+
| Zero-day test (Reconnaissance) |   13987 |
+--------------------------------+---------+

Features : 39
Zero-day samples are NOT present in training.


In [5]:
################################################################################
# Cell 3 — Representative samples via BGE-M3 embeddings
#
# Identical methodology to WUSTL notebook:
#   1. Subsample up to MAX_EMBED rows per class
#   2. Embed as stringified row lists using BAAI/bge-m3
#   3. Compute mean embedding
#   4. Select top N_REPR rows by cosine similarity to mean
################################################################################

import json
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from tqdm import tqdm

embeddings = HuggingFaceEmbeddings(
    model_name='BAAI/bge-m3',
    model_kwargs={'device': 'mps'},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 64}
)


def get_representative_samples_bge(
    df: pd.DataFrame, n: int = 10, max_embed: int = 100, seed: int = 42
) -> pd.DataFrame:
    """
    Subsample up to max_embed rows, embed using BGE-M3, compute mean embedding,
    return n rows with highest cosine similarity to the mean.
    """
    sample = df.sample(n=min(max_embed, len(df)), random_state=seed)
    docs   = [str(row.tolist()) for _, row in sample.iterrows()]
    vecs   = np.array(embeddings.embed_documents(docs))
    mean_vec = vecs.mean(axis=0)
    norms = np.linalg.norm(vecs, axis=1) * np.linalg.norm(mean_vec)
    sims  = (vecs @ mean_vec) / np.where(norms == 0, 1e-9, norms)
    top_idx = np.argsort(sims)[::-1][:n]
    return sample.iloc[top_idx]


print('Computing normal representative samples via BGE-M3...')
normal_repr = get_representative_samples_bge(
    normal_df_train, n=N_REPR, max_embed=MAX_EMBED, seed=SEED_MAIN
)

print('Computing attack representative samples via BGE-M3 (known attacks only)...')
attack_repr = get_representative_samples_bge(
    attack_df_train, n=N_REPR, max_embed=MAX_EMBED, seed=SEED_MAIN
)

print(f'\nRepresentative normal  samples : {len(normal_repr)}')
print(f'Representative attack  samples : {len(attack_repr)}')
print(f'Embedding model        : BAAI/bge-m3 (normalised cosine, batch=64)')
print(f'Max embedded per class : {MAX_EMBED}')

Computing normal representative samples via BGE-M3...
Computing attack representative samples via BGE-M3 (known attacks only)...

Representative normal  samples : 10
Representative attack  samples : 10
Embedding model        : BAAI/bge-m3 (normalised cosine, batch=64)
Max embedded per class : 100


In [6]:
################################################################################
# Cell 3b — Finalize anonymization maps and build entry dicts
#
# Produces four entry strings:
#   normal_entries_orig / attack_entries_orig  — real feature names
#   normal_entries_anon / attack_entries_anon  — anonymized f0…fn labels
################################################################################

anon_map, reverse_map = build_anon_maps(feature_cols)

normal_entries_orig = json.dumps({col: normal_repr[col].tolist() for col in feature_cols})
attack_entries_orig = json.dumps({col: attack_repr[col].tolist() for col in feature_cols})

normal_entries_anon = json.dumps({anon_map[col]: normal_repr[col].tolist() for col in feature_cols})
attack_entries_anon = json.dumps({anon_map[col]: attack_repr[col].tolist() for col in feature_cols})

print(f'Anonymization map built: {len(anon_map)} features')
print(f'Sample mapping: {list(anon_map.items())[:5]}')

Anonymization map built: 39 features
Sample mapping: [('dur', 'f0'), ('spkts', 'f1'), ('dpkts', 'f2'), ('sbytes', 'f3'), ('dbytes', 'f4')]


In [7]:
################################################################################
# Cell 4 — Anon-aware Policy Evaluation Tool
#
# Accepts both real names and f{i} labels via reverse_map lookup.
# try/except TypeError handles NaN rows in UNSW-NB15 float columns.
################################################################################

import operator as op_module
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
from statistics import mode
from typing import Annotated
from langchain_core.tools import tool

show_progress = False
operators = {
    '<':  op_module.lt,
    '>':  op_module.gt,
    '==': op_module.eq,
    '<=': op_module.le,
    '>=': op_module.ge,
    '!=': op_module.ne,
}


@tool
def evaluation_tool(
        feature_name: Annotated[str, 'Feature name (real or anonymous f{i})'],
        value: Annotated[float, 'Threshold value'],
        op: Annotated[str, 'Operator (<, >, <=, >=, ==, !=']
) -> float:
    """Evaluate a single threshold rule on training data. Returns macro F1-score."""
    real_name = reverse_map.get(feature_name, feature_name)
    try:
        value = float(value)
    except (ValueError, TypeError):
        pass
    if op not in operators:
        raise ValueError(f'Unsupported operator: {op}')
    datasets = {'normal': normal_df_train, 'attack': attack_df_train}
    y_pred, y_true = [], []
    for label, dataset in datasets.items():
        for i in tqdm(range(len(dataset)), disable=not show_progress,
                      ncols=100, desc=f'Evaluating {label}...'):
            y_true.append(label)
            try:
                y_pred.append('attack' if operators[op](dataset.iloc[i][real_name], value) else 'normal')
            except (KeyError, TypeError):
                y_pred.append('normal')
    report = classification_report(y_true, y_pred, digits=4, output_dict=True)
    return report['macro avg']['f1-score']


print('evaluation_tool (anon-aware) defined.')

evaluation_tool (anon-aware) defined.


In [ ]:
################################################################################
# Cell 5 — LangGraph Pipeline
#
# Identical graph topology to WUSTL notebook.
# Custom ToolNode replaces langgraph.prebuilt.ToolNode (unavailable in this version).
################################################################################

import dotenv, os, json
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
# from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langgraph.graph import MessagesState, StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display

dotenv.load_dotenv(os.getcwd() + '/../.env')


class State(MessagesState):
    i: int
    max_f1s: float
    best_tool_calls: list


llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.1)
# llm = ChatGoogleGenerativeAI(model='gemini-1.5-pro', temperature=0.1)
# llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0.1)

tools_list = [evaluation_tool]
llm_with_tools = llm.bind_tools(tools_list)


def extract_tool_calls(message):
    """Return normalized tool calls from either parsed or raw OpenAI format."""
    parsed = getattr(message, 'tool_calls', None)
    if parsed:
        return parsed
    raw_calls = (getattr(message, 'additional_kwargs', {}) or {}).get('tool_calls', [])
    normalized = []
    for call in raw_calls:
        fn = call.get('function', {})
        args = fn.get('arguments', {})
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except Exception:
                args = {}
        normalized.append({'id': call.get('id', ''), 'name': fn.get('name', ''), 'args': args})
    return normalized


# Custom ToolNode — replaces langgraph.prebuilt.ToolNode (unavailable in this version)
def make_tool_node(tools):
    tools_by_name = {t.name: t for t in tools}
    def tool_node(state):
        ai_msg = next(
            m for m in reversed(state['messages'])
            if extract_tool_calls(m)
        )
        results = []
        for tc in extract_tool_calls(ai_msg):
            result = tools_by_name[tc['name']].invoke(tc['args'])
            results.append(ToolMessage(content=str(result), tool_call_id=tc['id']))
        return {'messages': results}
    return tool_node


def llm_node(state):
    completion = llm_with_tools.invoke(state['messages'])
    return {'messages': [completion], 'i': state['i'] + 1}


def evaluate_node(state):
    """Evaluate current rules on known-attack test set and send feedback to LLM."""
    ai_messages = [
        m for m in state['messages']
        if isinstance(m, AIMessage) and extract_tool_calls(m)
    ]
    if not ai_messages:
        return {}
    tool_calls = extract_tool_calls(ai_messages[-1])

    datasets = {'normal': normal_df_test, 'attack': attack_df_test}
    y_pred, y_true = [], []
    for label, dataset in datasets.items():
        for i in tqdm(range(len(dataset)), disable=False, ncols=100,
                      desc=f'Test eval {label}...'):
            try:
                votes = [
                    'attack' if operators[tc['args']['op']](
                        dataset.iloc[i][
                            reverse_map.get(tc['args']['feature_name'],
                                            tc['args']['feature_name'])],
                        float(tc['args']['value'])
                    ) else 'normal'
                    for tc in tool_calls
                ]
                y_pred.append(mode(votes))
            except Exception:
                y_pred.append('normal')
            y_true.append(label)

    report = classification_report(y_true, y_pred, digits=4, output_dict=True)
    matrix = confusion_matrix(y_true, y_pred)
    f1     = report['macro avg']['f1-score']
    print(f'  Iter {state["i"]}: known-attack test macro-F1 = {f1:.4f}')
    print(matrix)

    new_max  = max(state['max_f1s'], f1)
    new_best = tool_calls if f1 >= state['max_f1s'] else state['best_tool_calls']

    feedback = HumanMessage(
        f'Current macro avg F1 on test set (known attacks only): {f1:.4f}. '
        f'Best so far: {new_max:.4f}. '
        f'If this is greater than the previous best, keep performing rules and revise underperforming ones. '
        f'Otherwise revise all rules to exceed the best. '
        f'Generate exactly {k} rules and make a tool call for each.'
    )
    return {
        'messages': [feedback],
        'max_f1s':  new_max,
        'best_tool_calls': new_best,
    }


def tools_condition_edge(state):
    return 'tools' if extract_tool_calls(state['messages'][-1]) else 'evaluate_node'


def feedback_condition_edge(state):
    return 'llm_node' if state['i'] < n else END


builder = StateGraph(State)
builder.add_node('llm_node', llm_node)
builder.add_node('tools', make_tool_node(tools_list))
builder.add_node('evaluate_node', evaluate_node)

builder.add_edge(START, 'llm_node')
builder.add_conditional_edges('llm_node', tools_condition_edge, ['tools', 'evaluate_node'])
builder.add_edge('tools', 'llm_node')
builder.add_conditional_edges('evaluate_node', feedback_condition_edge, ['llm_node', END])

graph = builder.compile(checkpointer=MemorySaver())
display(Image(graph.get_graph().draw_mermaid_png()))
print('Graph compiled.')

In [9]:
################################################################################
# Cell 6a — Run pipeline: ORIGINAL feature names
################################################################################

import dotenv, os
from langchain_core.messages import SystemMessage, HumanMessage

dotenv.load_dotenv(os.getcwd() + '/../.env')

system_message = SystemMessage(
    f"""You are a skilled security data analyst.
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Your task is to generate exactly {k} simple and deterministic rules for the top {k} important features to filter attack entries.
Supported operators: >, <, >=, <=
NEVER use '==' or '!=' - these are forbidden for numeric features.
Generate exactly {k} rules and make a tool call for each rule."""
)

print('--- ORIGINAL feature names ---')
initial_state_orig = State(
    i=0, max_f1s=0.5, best_tool_calls=[],
    messages=[
        system_message,
        HumanMessage(
            f'Analyze the following network data and generate {k} rules to identify attack entries.\n\n'
            f'Normal Entries:\n```{normal_entries_orig}```\n\n'
            f'Attack Entries:\n```{attack_entries_orig}```'
        )
    ]
)
config_orig = {
    'configurable': {'thread_id': f'anon-orig-unsw-nb15-{WITHHELD_CLASS}-seed{SEED_MAIN}'},
    'recursion_limit': 100
}
graph_orig = builder.compile(checkpointer=MemorySaver())
output_orig = graph_orig.invoke(initial_state_orig, config_orig)

best_tool_calls_orig = output_orig['best_tool_calls']
best_f1_orig = output_orig['max_f1s']
print(f'\nBest known-attack macro-F1 (original): {best_f1_orig:.4f}')
print(f'Rules ({len(best_tool_calls_orig)}):')
for tc in best_tool_calls_orig:
    a = tc['args']
    print(f'  {a["feature_name"]:25s} {a["op"]:2s} {a["value"]}')

--- ORIGINAL feature names ---


Test eval attack...: 100%|█████████████████████████████████| 18600/18600 [00:01<00:00, 12446.59it/s]


  Iter 2: known-attack test macro-F1 = 0.6777
[[12042  6558]
 [ 5420 13180]]


Test eval attack...: 100%|█████████████████████████████████| 18600/18600 [00:01<00:00, 12233.00it/s]


  Iter 4: known-attack test macro-F1 = 0.7547
[[12020  6580]
 [ 2431 16169]]


Test eval attack...: 100%|█████████████████████████████████| 18600/18600 [00:01<00:00, 12511.68it/s]


  Iter 6: known-attack test macro-F1 = 0.7547
[[12020  6580]
 [ 2431 16169]]

Best known-attack macro-F1 (original): 0.7547
Rules (5):
  dur                       <  0.005
  spkts                     <  5
  dpkts                     <  1
  rate                      >  90000
  dttl                      <  1


In [10]:
################################################################################
# Cell 6b — Run pipeline: ANONYMIZED feature names
################################################################################

print('--- ANONYMIZED feature names ---')
initial_state_anon = State(
    i=0, max_f1s=0.5, best_tool_calls=[],
    messages=[
        system_message,
        HumanMessage(
            f'Analyze the following network data and generate {k} rules to identify attack entries.\n\n'
            f'Normal Entries:\n```{normal_entries_anon}```\n\n'
            f'Attack Entries:\n```{attack_entries_anon}```'
        )
    ]
)
config_anon = {
    'configurable': {'thread_id': f'anon-anon-unsw-nb15-{WITHHELD_CLASS}-seed{SEED_MAIN}'},
    'recursion_limit': 100
}
graph_anon = builder.compile(checkpointer=MemorySaver())
output_anon = graph_anon.invoke(initial_state_anon, config_anon)

best_tool_calls_anon = output_anon['best_tool_calls']
best_f1_anon = output_anon['max_f1s']
print(f'\nBest known-attack macro-F1 (anon): {best_f1_anon:.4f}')
print(f'Rules ({len(best_tool_calls_anon)}) — shown with real names:')
for tc in best_tool_calls_anon:
    a = tc['args']
    real = reverse_map.get(a['feature_name'], a['feature_name'])
    print(f'  {a["feature_name"]:6s} ({real:20s}) {a["op"]:2s} {a["value"]}')

--- ANONYMIZED feature names ---


Test eval attack...: 100%|█████████████████████████████████| 18600/18600 [00:01<00:00, 12569.43it/s]


  Iter 2: known-attack test macro-F1 = 0.6637
[[12070  6530]
 [ 5979 12621]]


Test eval attack...: 100%|██████████████████████████████████| 18600/18600 [00:02<00:00, 9271.07it/s]


  Iter 4: known-attack test macro-F1 = 0.6630
[[12085  6515]
 [ 6021 12579]]


Test eval attack...: 100%|█████████████████████████████████| 18600/18600 [00:01<00:00, 12241.23it/s]


  Iter 6: known-attack test macro-F1 = 0.6827
[[12067  6533]
 [ 5255 13345]]

Best known-attack macro-F1 (anon): 0.6827
Rules (5) — shown with real names:
  f0     (dur                 ) <  0.017
  f1     (spkts               ) <  7
  f2     (dpkts               ) <  3
  f4     (dbytes              ) <  260
  f5     (rate                ) >  4900


In [11]:
################################################################################
# Cell 7 — ZDR and FPR_benign for both conditions
################################################################################

from statistics import mode

def evaluate_zero_day_anon(tool_calls, zd_df):
    """Like evaluate_zero_day but resolves anon feature names via reverse_map."""
    preds = []
    for i in tqdm(range(len(zd_df)), ncols=100,
                  desc=f'Zero-day eval [{WITHHELD_CLASS}]...'):
        try:
            votes = [
                'attack' if operators[tc['args']['op']](
                    zd_df.iloc[i][reverse_map.get(tc['args']['feature_name'],
                                                   tc['args']['feature_name'])],
                    float(tc['args']['value'])
                ) else 'normal'
                for tc in tool_calls
            ]
            preds.append(mode(votes))
        except Exception:
            preds.append('normal')
    zdr = sum(p == 'attack' for p in preds) / max(len(preds), 1)
    return zdr, preds


def fpr_and_precision_anon(tool_calls, benign_df, tp_count):
    """Compute FPR_benign and precision_reconn. Resolves anon names."""
    bp = []
    for i in tqdm(range(len(benign_df)), ncols=100, desc='FPR_benign...'):
        try:
            votes = [
                'attack' if operators[tc['args']['op']](
                    benign_df.iloc[i][reverse_map.get(tc['args']['feature_name'],
                                                       tc['args']['feature_name'])],
                    float(tc['args']['value'])
                ) else 'normal'
                for tc in tool_calls
            ]
            bp.append(mode(votes))
        except Exception:
            bp.append('normal')
    fp_b  = sum(p == 'attack' for p in bp)
    fpr_b = fp_b / max(len(benign_df), 1)
    prec  = tp_count / (tp_count + fp_b) if (tp_count + fp_b) > 0 else 0.0
    return fpr_b, prec, fp_b


# Original
zdr_orig, preds_orig = evaluate_zero_day_anon(best_tool_calls_orig, zeroday_df_test)
tp_orig = sum(p == 'attack' for p in preds_orig)
fpr_orig, prec_orig, fp_orig = fpr_and_precision_anon(best_tool_calls_orig, normal_df_test, tp_orig)

# Anonymized
zdr_anon, preds_anon = evaluate_zero_day_anon(best_tool_calls_anon, zeroday_df_test)
tp_anon = sum(p == 'attack' for p in preds_anon)
fpr_anon, prec_anon, fp_anon = fpr_and_precision_anon(best_tool_calls_anon, normal_df_test, tp_anon)

assert 0 <= zdr_orig <= 1 and 0 <= zdr_anon <= 1, 'ZDR out of bounds'

print(f'\n=== Condition Comparison (seed {SEED_MAIN}) ===')
print(f'Condition    ZDR       FPR_benign  Precision_reconn')
print(f'Original     {zdr_orig:.4f}    {fpr_orig:.4f}      {prec_orig:.4f}')
print(f'Anonymized   {zdr_anon:.4f}    {fpr_anon:.4f}      {prec_anon:.4f}')
print(f'\nΔZDR (orig − anon): {zdr_orig - zdr_anon:+.4f}')

FPR_benign...: 100%|███████████████████████████████████████| 18600/18600 [00:01<00:00, 11866.45it/s]


=== Condition Comparison (seed 42) ===
Condition    ZDR       FPR_benign  Precision_reconn
Original     0.4820    0.1307      0.7350
Anonymized   0.4833    0.2825      0.5626

ΔZDR (orig − anon): -0.0013


In [12]:
################################################################################
# Cell 8 — ML baseline comparison
################################################################################

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_train   = train_df[feature_cols].values
y_train   = train_df['binary_label'].values
X_test    = test_known_df[feature_cols].values
y_test    = test_known_df['binary_label'].values
X_zd      = zeroday_df_test.values
X_benign  = normal_df_test.values

ml_results = []
for name, model in [
    ('Decision Tree', DecisionTreeClassifier(random_state=SEED_MAIN)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=SEED_MAIN, n_jobs=-1)),
]:
    model.fit(X_train, y_train)
    rep = classification_report(model.predict(X_test), y_test, output_dict=True)
    known_f1  = rep['macro avg']['f1-score']
    y_zd      = model.predict(X_zd)
    y_ben     = model.predict(X_benign)
    ml_zdr    = (y_zd  == 'attack').mean()
    fp_b      = (y_ben == 'attack').sum()
    fpr_b     = fp_b / max(len(X_benign), 1)
    tp_zd     = (y_zd  == 'attack').sum()
    prec_zd   = tp_zd / (tp_zd + fp_b) if (tp_zd + fp_b) > 0 else 0.0
    ml_results.append([name, f'{known_f1:.4f}', f'{ml_zdr:.4f}', f'{fpr_b:.4f}', f'{prec_zd:.4f}'])
    print(f'{name:20s}  ZDR={ml_zdr:.4f}  FPR_benign={fpr_b:.4f}  Prec_reconn={prec_zd:.4f}')

Decision Tree         ZDR=0.8454  FPR_benign=0.0679  Prec_reconn=0.9035
Random Forest         ZDR=0.8787  FPR_benign=0.0411  Prec_reconn=0.9414


In [13]:
################################################################################
# Cell 9 — Full comparison table + save
################################################################################

import datetime, os
from tabulate import tabulate

all_results = [
    ['LLM (original names)',  f'{best_f1_orig:.4f}', f'{zdr_orig:.4f}  ({zdr_orig*100:.1f}%)',
     f'{fpr_orig:.4f}  ({fpr_orig*100:.1f}%)', f'{prec_orig:.4f}'],
    ['LLM (anonymized)',      f'{best_f1_anon:.4f}', f'{zdr_anon:.4f}  ({zdr_anon*100:.1f}%)',
     f'{fpr_anon:.4f}  ({fpr_anon*100:.1f}%)', f'{prec_anon:.4f}'],
] + ml_results

print(f'\n=== ANONYMIZED ZERO-DAY EXPERIMENT RESULTS ===')
print(f'Dataset  : {DATASET_NAME}')
print(f'Withheld : {WITHHELD_CLASS}  ({len(zeroday_df_test):,} zero-day test samples)')
print(f'Seed     : {SEED_MAIN}')
print()
print(tabulate(
    all_results,
    headers=['Method', 'Known-Attack F1', f'ZDR — {WITHHELD_CLASS}', 'FPR_benign', 'Precision_reconn'],
    tablefmt='grid'
))

os.makedirs('results/anon', exist_ok=True)
ts = datetime.datetime.now().strftime('%Y-%m-%d-%H-%M-%S')
out = {
    'dataset': DATASET_NAME, 'withheld_class': WITHHELD_CLASS, 'seed': SEED_MAIN,
    'zero_day_samples': len(zeroday_df_test),
    'original':   {'known_f1': best_f1_orig, 'zdr': zdr_orig, 'fpr_benign': fpr_orig,
                   'precision_reconn': prec_orig,
                   'rules': [tc['args'] for tc in best_tool_calls_orig]},
    'anonymized': {'known_f1': best_f1_anon, 'zdr': zdr_anon, 'fpr_benign': fpr_anon,
                   'precision_reconn': prec_anon,
                   'rules': [{'feature_name': tc['args']['feature_name'],
                               'real_name': reverse_map.get(tc['args']['feature_name'], tc['args']['feature_name']),
                               'value': tc['args']['value'], 'op': tc['args']['op']}
                              for tc in best_tool_calls_anon]},
    'delta_zdr': zdr_orig - zdr_anon,
}
out_path = f'results/anon/zeroday-{WITHHELD_CLASS}-anon-ablation-seed{SEED_MAIN}-{ts}.json'
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f'\nResults saved to {out_path}')
print(f'\nΔZDR (original − anonymized): {zdr_orig - zdr_anon:+.4f}')


=== ANONYMIZED ZERO-DAY EXPERIMENT RESULTS ===
Dataset  : unsw-nb15
Withheld : Reconnaissance  (13,987 zero-day test samples)
Seed     : 42

+----------------------+-------------------+------------------------+-----------------+--------------------+
| Method               |   Known-Attack F1 | ZDR — Reconnaissance   | FPR_benign      |   Precision_reconn |
+======================+===================+========================+=================+====================+
| LLM (original names) |            0.7547 | 0.4820  (48.2%)        | 0.1307  (13.1%) |             0.735  |
+----------------------+-------------------+------------------------+-----------------+--------------------+
| LLM (anonymized)     |            0.6827 | 0.4833  (48.3%)        | 0.2825  (28.3%) |             0.5626 |
+----------------------+-------------------+------------------------+-----------------+--------------------+
| Decision Tree        |            0.9306 | 0.8454                 | 0.0679          |        

In [15]:
################################################################################
# Cell 10 — Multi-seed loop
#
# Re-runs Cells 6a, 6b, 7 for each seed in SEEDS.
# For each seed: re-samples balanced pool, re-splits 80/20, recomputes repr
# samples, runs both pipeline conditions, evaluates ZDR + FPR.
# Aggregates mean ± std across seeds.
################################################################################

import numpy as np
from langchain_core.messages import SystemMessage, HumanMessage

seed_records = []

for seed in SEEDS:
    print(f"\n{'='*60}")
    print(f'SEED {seed}')
    print(f"{'='*60}")

    # --- Re-sample balanced pool ---
    b_samp = benign_df.sample(n=target_count, random_state=seed)
    a_samp = known_atk_df.sample(n=target_count, random_state=seed)
    pool_s = pd.concat([b_samp, a_samp], ignore_index=True)
    pool_s['binary_label'] = pool_s[LABEL_COL].apply(
        lambda x: 'normal' if x == BENIGN_CLASS else 'attack'
    )

    tr_idx, te_idx = train_test_split(
        pool_s.index, test_size=0.2, random_state=seed,
        stratify=pool_s['binary_label']
    )
    tr_s = pool_s.loc[tr_idx]
    te_s = pool_s.loc[te_idx]

    norm_tr_s = tr_s[tr_s['binary_label'] == 'normal'][feature_cols].reset_index(drop=True)
    atk_tr_s  = tr_s[tr_s['binary_label'] == 'attack'][feature_cols].reset_index(drop=True)
    norm_te_s = te_s[te_s['binary_label'] == 'normal'][feature_cols].reset_index(drop=True)
    atk_te_s  = te_s[te_s['binary_label'] == 'attack'][feature_cols].reset_index(drop=True)

    # Override globals used by evaluation_tool and evaluate_node
    normal_df_train = norm_tr_s
    attack_df_train = atk_tr_s
    normal_df_test  = norm_te_s
    attack_df_test  = atk_te_s

    # --- Recompute representative samples ---
    nr_s = get_representative_samples_bge(norm_tr_s, n=N_REPR, max_embed=MAX_EMBED, seed=seed)
    ar_s = get_representative_samples_bge(atk_tr_s,  n=N_REPR, max_embed=MAX_EMBED, seed=seed)

    ne_orig_s = json.dumps({col: nr_s[col].tolist() for col in feature_cols})
    ae_orig_s = json.dumps({col: ar_s[col].tolist() for col in feature_cols})
    ne_anon_s = json.dumps({anon_map[col]: nr_s[col].tolist() for col in feature_cols})
    ae_anon_s = json.dumps({anon_map[col]: ar_s[col].tolist() for col in feature_cols})

    sys_msg = SystemMessage(
        f"""You are a skilled security data analyst.
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Your task is to generate exactly {k} simple and deterministic rules for the top {k} important features to filter attack entries.
Supported operators: >, <, >=, <=
NEVER use '==' or '!=' - these are forbidden for numeric features.
Generate exactly {k} rules and make a tool call for each rule."""
    )

    # --- 6a: Original ---
    print(f'  Running ORIGINAL (seed={seed})...')
    st_orig_s = State(
        i=0, max_f1s=0.5, best_tool_calls=[],
        messages=[
            sys_msg,
            HumanMessage(
                f'Analyze the following network data and generate {k} rules to identify attack entries.\n\n'
                f'Normal Entries:\n```{ne_orig_s}```\n\nAttack Entries:\n```{ae_orig_s}```'
            )
        ]
    )
    cfg_orig_s = {
        'configurable': {'thread_id': f'anon-orig-unsw-nb15-{WITHHELD_CLASS}-seed{seed}'},
        'recursion_limit': 100
    }
    g_orig_s = builder.compile(checkpointer=MemorySaver())
    out_orig_s = g_orig_s.invoke(st_orig_s, cfg_orig_s)
    tc_orig_s = out_orig_s['best_tool_calls']

    # --- 6b: Anonymized ---
    print(f'  Running ANONYMIZED (seed={seed})...')
    st_anon_s = State(
        i=0, max_f1s=0.5, best_tool_calls=[],
        messages=[
            sys_msg,
            HumanMessage(
                f'Analyze the following network data and generate {k} rules to identify attack entries.\n\n'
                f'Normal Entries:\n```{ne_anon_s}```\n\nAttack Entries:\n```{ae_anon_s}```'
            )
        ]
    )
    cfg_anon_s = {
        'configurable': {'thread_id': f'anon-anon-unsw-nb15-{WITHHELD_CLASS}-seed{seed}'},
        'recursion_limit': 100
    }
    g_anon_s = builder.compile(checkpointer=MemorySaver())
    out_anon_s = g_anon_s.invoke(st_anon_s, cfg_anon_s)
    tc_anon_s = out_anon_s['best_tool_calls']

    # --- 7: ZDR + FPR ---
    zdr_o_s, pr_o_s = evaluate_zero_day_anon(tc_orig_s, zeroday_df_test)
    tp_o_s = sum(p == 'attack' for p in pr_o_s)
    fpr_o_s, prec_o_s, _ = fpr_and_precision_anon(tc_orig_s, norm_te_s, tp_o_s)

    zdr_a_s, pr_a_s = evaluate_zero_day_anon(tc_anon_s, zeroday_df_test)
    tp_a_s = sum(p == 'attack' for p in pr_a_s)
    fpr_a_s, prec_a_s, _ = fpr_and_precision_anon(tc_anon_s, norm_te_s, tp_a_s)

    seed_records.append({
        'seed': seed,
        'zdr_orig': zdr_o_s,  'fpr_orig': fpr_o_s,  'prec_orig': prec_o_s,
        'zdr_anon': zdr_a_s,  'fpr_anon': fpr_a_s,  'prec_anon': prec_a_s,
        'delta_zdr': zdr_o_s - zdr_a_s,
        'f1_orig': out_orig_s['max_f1s'],
        'f1_anon': out_anon_s['max_f1s'],
    })
    print(f'  Seed {seed}: ZDR_orig={zdr_o_s:.4f}  ZDR_anon={zdr_a_s:.4f}  ΔZDR={zdr_o_s - zdr_a_s:+.4f}')

# Restore globals to seed-42 values for downstream use
normal_df_train = train_df[train_df['binary_label'] == 'normal'][feature_cols].reset_index(drop=True)
attack_df_train = train_df[train_df['binary_label'] == 'attack'][feature_cols].reset_index(drop=True)
normal_df_test  = test_known_df[test_known_df['binary_label'] == 'normal'][feature_cols].reset_index(drop=True)
attack_df_test  = test_known_df[test_known_df['binary_label'] == 'attack'][feature_cols].reset_index(drop=True)

# --- Aggregate ---
zdrs_o  = [r['zdr_orig']  for r in seed_records]
zdrs_a  = [r['zdr_anon']  for r in seed_records]
fprs_o  = [r['fpr_orig']  for r in seed_records]
fprs_a  = [r['fpr_anon']  for r in seed_records]
deltas  = [r['delta_zdr'] for r in seed_records]

print(f'\n=== MULTI-SEED SUMMARY (seeds={SEEDS}) ===')
print(f'Dataset  : {DATASET_NAME}')
print(f'Withheld : {WITHHELD_CLASS}')
print()

summary_rows = []
for r in seed_records:
    summary_rows.append([
        r['seed'],
        f"{r['zdr_orig']:.4f}", f"{r['fpr_orig']:.4f}",
        f"{r['zdr_anon']:.4f}", f"{r['fpr_anon']:.4f}",
        f"{r['delta_zdr']:+.4f}",
    ])
summary_rows.append([
    'mean±std',
    f"{np.mean(zdrs_o):.4f}±{np.std(zdrs_o):.4f}",
    f"{np.mean(fprs_o):.4f}±{np.std(fprs_o):.4f}",
    f"{np.mean(zdrs_a):.4f}±{np.std(zdrs_a):.4f}",
    f"{np.mean(fprs_a):.4f}±{np.std(fprs_a):.4f}",
    f"{np.mean(deltas):+.4f}±{np.std(deltas):.4f}",
])
print(tabulate(
    summary_rows,
    headers=['Seed', 'ZDR_orig', 'FPR_orig', 'ZDR_anon', 'FPR_anon', 'ΔZDR'],
    tablefmt='grid'
))

os.makedirs('results/anon', exist_ok=True)
multiseed_out = {
    'dataset': DATASET_NAME,
    'withheld_class': WITHHELD_CLASS,
    'seeds': SEEDS,
    'per_seed': seed_records,
    'aggregate': {
        'zdr_orig_mean': float(np.mean(zdrs_o)), 'zdr_orig_std': float(np.std(zdrs_o)),
        'zdr_anon_mean': float(np.mean(zdrs_a)), 'zdr_anon_std': float(np.std(zdrs_a)),
        'fpr_orig_mean': float(np.mean(fprs_o)), 'fpr_orig_std': float(np.std(fprs_o)),
        'fpr_anon_mean': float(np.mean(fprs_a)), 'fpr_anon_std': float(np.std(fprs_a)),
        'delta_zdr_mean': float(np.mean(deltas)), 'delta_zdr_std': float(np.std(deltas)),
    }
}
ms_path = f'results/anon/zeroday-{WITHHELD_CLASS}-anon-multiseed.json'
with open(ms_path, 'w') as f:
    json.dump(multiseed_out, f, indent=2)
print(f'\nMulti-seed results saved to {ms_path}')
print(f'Mean ΔZDR (orig − anon): {np.mean(deltas):+.4f} ± {np.std(deltas):.4f}')


SEED 42
  Running ORIGINAL (seed=42)...


Test eval attack...: 100%|██████████████████████████████████| 18600/18600 [00:01<00:00, 9456.76it/s]


  Iter 2: known-attack test macro-F1 = 0.6962
[[12042  6558]
 [ 4714 13886]]


Test eval attack...: 100%|█████████████████████████████████| 18600/18600 [00:01<00:00, 11697.70it/s]


  Iter 4: known-attack test macro-F1 = 0.6495
[[12169  6431]
 [ 6607 11993]]


...

  Iter 4: known-attack test macro-F1 = 0.6948
[[11859  6741]
 [ 4575 14025]]


Test eval attack...: 100%|█████████████████████████████████| 18600/18600 [00:01<00:00, 11568.36it/s]


  Iter 6: known-attack test macro-F1 = 0.6970
[[11853  6747]
 [ 4482 14118]]


FPR_benign...: 100%|███████████████████████████████████████| 18600/18600 [00:01<00:00, 12109.01it/s]


  Seed 456: ZDR_orig=0.4983  ZDR_anon=0.4799  ΔZDR=+0.0184

=== MULTI-SEED SUMMARY (seeds=[42, 123, 456]) ===
Dataset  : unsw-nb15
Withheld : Reconnaissance

+----------+---------------+---------------+---------------+---------------+----------------+
| Seed     | ZDR_orig      | FPR_orig      | ZDR_anon      | FPR_anon      | ΔZDR           |
+==========+===============+===============+===============+===============+================+
| 42       | 0.4829        | 0.2534        | 0.4836        | 0.3215        | -0.0007        |
+----------+---------------+---------------+---------------+---------------+----------------+
| 123      | 0.4779        | 0.0845        | 0.4799        | 0.1217        | -0.0021        |
+----------+---------------+---------------+---------------+---------------+----------------+
| 456      | 0.4983        | 0.1380        | 0.4799        | 0.2410        | +0.0184        |
+----------+---------------+---------------+---------------+---------------+--------------